# Retinopathy of Prematurity (ROP) Classification
### Binary Classification of Retinal Images — Normal vs Abnormal

---

**Model:** EfficientNet-B0 (Transfer Learning)  
**Task:** Binary Image Classification  
**Classes:** Normal Retina (DG0) · Abnormal Retina (DG1)  
**Framework:** PyTorch  
**Hardware:** Apple MPS (Metal Performance Shaders)

---

## Table of Contents

1. [Dataset Overview](#step-1)
2. [Data Preparation](#step-2)
3. [Label Assignment — Binary Classification](#step-3)
4. [Label Conversion Result](#step-4)
5. [Patient-ID Based Data Split](#step-5)
6. [Model Architecture & Training](#step-6)
7. [Training Results](#training-results)
8. [Evaluation on Test Set](#evaluation)
9. [Single Image Prediction](#prediction)
10. [Summary](#summary)

---
## Step 1 — Dataset Overview <a id='step-1'></a>

The original dataset (`image_stack_without_caption`) consists of retinal images organized across **11 disease-grade (DG) classes**. The distribution is heavily imbalanced, with DG0 (normal) accounting for nearly half of all images.

| DG Class | Image Count | Category |
|:--------:|:-----------:|:--------:|
| DG 0  | 2,980 | Normal   |
| DG 1  |    45 | Abnormal |
| DG 2  |   252 | Abnormal |
| DG 3  |   458 | Abnormal |
| DG 4  |   125 | Abnormal |
| DG 8  |   470 | Abnormal |
| DG 9  |   379 | Abnormal |
| DG 10 |    72 | Abnormal |
| DG 11 | 1,061 | Abnormal |
| DG 12 |    48 | Abnormal |
| DG 13 |   114 | Abnormal |
| **Total** | **6,004** | |

> **Observation:** DG0 contains 2,980 images (~49.6%), while the remaining 10 abnormal classes together make up 3,024 images (~50.4%). This near-50/50 split at the binary level is well-suited for classification after label consolidation.

---
## Step 2 — Data Preparation <a id='step-2'></a>

The entire original dataset is copied from `image_stack_without_caption/` into a new working directory `rop_dataset_working_copy/` to preserve the source data intact.

```
image_stack_without_caption/    ←  original (read-only)
        │
        └──  copy  ──►   rop_dataset_working_copy/   ←  working directory
```

All subsequent processing, relabeling, and splitting is performed on `rop_dataset_working_copy/`.

---
## Step 3 — Label Assignment: Binary Classification <a id='step-3'></a>

All 11 original DG classes are collapsed into two binary categories:

| Label | Class | Rule |
|:-----:|:-----:|:----:|
| `DG0` | **Normal**   | Images originally labeled as DG 0 |
| `DG1` | **Abnormal** | All other DG classes (DG 1–13) merged |

```
DG 0          →  Normal   (DG0)
DG 1, 2, 3, 4,
DG 8, 9, 10,
DG 11, 12, 13 →  Abnormal (DG1)
```

This simplification is clinically meaningful: DG0 represents a healthy retina, while any non-zero grade indicates pathological changes requiring clinical attention.

---
## Step 4 — Label Conversion Result <a id='step-4'></a>

After reassigning all disease grades into the two binary classes, the dataset distribution becomes:

| Class | Count | Label |
|:-----:|:-----:|:-----:|
| DG0 | 2,980 | Normal   |
| DG1 | 3,024 | Abnormal |
| **Total** | **6,004** | |

> **Result:** The binary dataset is nearly perfectly balanced — 49.6% Normal vs 50.4% Abnormal — which is ideal for training a binary classifier without severe bias.

---
## Step 5 — Patient-ID Based Data Split <a id='step-5'></a>

The dataset is split using **patient IDs** (not random image sampling) to prevent data leakage between splits. All images from the same patient are kept within the same split.

### Split Strategy

| Split      | Ratio | DG0 included | DG1 included |
|:----------:|:-----:|:------------:|:------------:|
| Training   | 70%   | Same patient ID | Same patient ID |
| Validation | 15%   | Same patient ID | Same patient ID |
| Test       | 15%   | Same patient ID | Same patient ID |

### Split Results

| Split      | Images | Patients |
|:----------:|:------:|:--------:|
| Train      | 3,574  | 131      |
| Validation |   928  |  28      |
| Test       | 1,502  |  29      |
| **Total**  | **6,004** | **188** |

> **Why patient-level split?** Splitting by patient ID ensures that the model is tested on **entirely unseen patients**, not just unseen images from known patients. This gives a realistic measure of generalization to new clinical cases.

---
## Step 6 — Model Architecture & Training Setup <a id='step-6'></a>

### Task
- **Input:** Retinal fundus image (RGB)
- **Output:** Binary classification — `Normal (DG0)` or `Abnormal (DG1)`

### Model: EfficientNet-B0 (Transfer Learning)

```python
model = models.efficientnet_b0(weights="DEFAULT")
model.classifier = nn.Sequential(
    nn.Dropout(p=0.4),
    nn.Linear(in_features, 2)
)
```

The pre-trained EfficientNet-B0 backbone is used with a custom classification head. Dropout (p=0.4) is added for regularization.

### Training Configuration

| Hyperparameter    | Value |
|:-----------------:|:-----:|
| Image Size        | 160 × 160 |
| Batch Size        | 16 |
| Epochs            | 15 (with early stopping) |
| Learning Rate     | 5e-5 |
| Optimizer         | AdamW (weight decay = 1e-4) |
| Loss Function     | CrossEntropyLoss (class-weighted) |
| LR Scheduler      | ReduceLROnPlateau (factor=0.5, patience=2) |
| Early Stopping    | Patience = 5 epochs |

### Data Augmentation (Training Only)

| Augmentation      | Parameters |
|:-----------------:|:----------:|
| Random Horizontal Flip | — |
| Random Rotation        | ±15° |
| Color Jitter           | brightness/contrast/saturation = 0.2 |
| Random Affine          | translate = (0.05, 0.05), scale = (0.95, 1.05) |
| Normalize              | mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225] |

### Class Weights

To address any residual class imbalance, class weights are computed and applied in the loss function:

| Class | Count | Weight |
|:-----:|:-----:|:------:|
| DG0 (Normal)   | 2,002 | 1.785 |
| DG1 (Abnormal) | 1,572 | 2.274 |

---
## Training Results <a id='training-results'></a>

### Run 1 — Baseline (5 Epochs)

Initial exploratory run with 5 epochs using MobileNet-V3-Small to establish a baseline.

| Epoch | Train Loss | Train Acc | Val Loss | Val Acc |
|:-----:|:----------:|:---------:|:--------:|:-------:|
| 1 | 0.5764 | 69.11% | 1.5939 | 32.65% |
| 2 | 0.4225 | 81.17% | 0.9992 | 55.71% |
| 3 | 0.3420 | 85.70% | 0.9285 | 62.72% |
| 4 | 0.3075 | 87.16% | 0.9129 | 67.35% |
| 5 | 0.2460 | 90.01% | 1.0275 | 65.84% |

> **Best Validation Accuracy: 67.35%**  
> The gap between training accuracy (90%) and validation accuracy (67%) suggests overfitting beginning from epoch 5.

---

### Run 2 — Full Training (15 Epochs, Early Stopping)

Upgraded to EfficientNet-B0 with early stopping (patience = 5). Training halted at epoch 9.

| Epoch | Train Loss | Train Acc | Val Loss | Val Acc | Best |
|:-----:|:----------:|:---------:|:--------:|:-------:|:----:|
| 1  | 0.5638 | 70.03% | 0.6859 | 64.33% | ✓ |
| 2  | 0.4029 | 82.40% | 0.6540 | 69.40% | ✓ |
| 3  | 0.3178 | 87.05% | 0.8655 | 67.24% | |
| 4  | 0.2738 | 88.56% | 0.8855 | 70.58% | ✓ |
| 5  | 0.2293 | 90.77% | 1.1101 | 65.41% | |
| 6  | 0.2217 | 91.13% | 1.2277 | 67.03% | |
| 7  | 0.2072 | 91.10% | 1.2434 | 67.35% | |
| 8  | 0.1687 | 93.56% | 1.1468 | 69.29% | |
| 9  | 0.1567 | 94.01% | 1.3220 | 68.64% | |
| **Early Stop** | — | — | — | — | |

> **Best Validation Accuracy: 70.58%** (saved at epoch 4)

---

### Run 3 — Reproducibility Check

A second full training run was conducted to verify consistency. Early stopping triggered again at epoch 9.

| Epoch | Train Loss | Train Acc | Val Loss | Val Acc | Best |
|:-----:|:----------:|:---------:|:--------:|:-------:|:----:|
| 1  | 0.5652 | 70.20% | 0.6304 | 65.73% | ✓ |
| 2  | 0.4015 | 82.46% | 0.7402 | 67.03% | ✓ |
| 3  | 0.3347 | 86.46% | 0.8197 | 66.38% | |
| 4  | 0.2917 | 87.77% | 0.7771 | 69.18% | ✓ |
| 5  | 0.2460 | 90.29% | 1.0563 | 67.78% | |
| 6  | 0.2111 | 91.19% | 1.1268 | 67.78% | |
| 7  | 0.1935 | 91.91% | 1.0717 | 69.07% | |
| 8  | 0.1723 | 93.14% | 1.2266 | 66.70% | |
| 9  | 0.1681 | 94.07% | 1.3668 | 67.46% | |
| **Early Stop** | — | — | — | — | |

> **Best Validation Accuracy: 69.18%** (saved at epoch 4)

Both runs confirm the model consistently peaks around **epoch 4** and then begins to overfit, with early stopping correctly intervening around **epoch 9**.

---
## Evaluation on Test Set <a id='evaluation'></a>

The best saved model (`best_rop_model.pth`) was evaluated on the held-out test set of **1,502 images** (29 unseen patients).

### Metrics

| Metric    | Score   |
|:---------:|:-------:|
| Accuracy  | 71.17%  |
| Precision | 83.42%  |
| Recall    | 59.22%  |
| F1 Score  | 69.27%  |
| ROC-AUC   | 78.12%  |

### Confusion Matrix

```
                 Predicted Normal    Predicted Abnormal
Actual Normal         581                  97
Actual Abnormal       336                 488
```

| | Predicted Normal (DG0) | Predicted Abnormal (DG1) |
|:--|:---------------------:|:------------------------:|
| **Actual Normal (DG0)**   | 581 (True Negative)  |  97 (False Positive) |
| **Actual Abnormal (DG1)** | 336 (False Negative) | 488 (True Positive)  |

### Per-Class Classification Report

| Class    | Precision | Recall | F1-Score | Support |
|:--------:|:---------:|:------:|:--------:|:-------:|
| DG0 (Normal)   | 0.63 | 0.86 | 0.73 | 678 |
| DG1 (Abnormal) | 0.83 | 0.59 | 0.69 | 824 |
| **Macro Avg**  | 0.73 | 0.72 | 0.71 | 1502 |
| **Weighted Avg** | 0.74 | 0.71 | 0.71 | 1502 |

### Analysis

> - **High Precision for Abnormal (83.42%):** When the model predicts abnormal, it is correct 83% of the time — low false alarm rate.
> - **Lower Recall for Abnormal (59.22%):** The model misses ~41% of actual abnormal cases (336 false negatives). In a clinical setting, this is the most critical error type.
> - **ROC-AUC of 78.12%** indicates reasonable discriminative power between classes.
> - The model is more conservative in predicting abnormal, which increases precision but reduces recall.

---
## Single Image Prediction <a id='prediction'></a>

The trained model was tested on a single retinal image (`image.png`) using `predict.py`.

### Result

| Field | Value |
|:-----:|:-----:|
| **Prediction**  | Abnormal Retina (DG1) |
| **Confidence**  | 97.01% |

### Class Probabilities

| Class | Probability |
|:-----:|:-----------:|
| Normal Retina (DG0)   |  2.99% |
| Abnormal Retina (DG1) | 97.01% |

> The model predicted **Abnormal** with **97.01% confidence** on the test image, demonstrating high decisiveness in this case.

---
## Summary <a id='summary'></a>

### Project Pipeline

```
Raw Dataset (11 DG classes, 6,004 images)
        │
        ▼
Binary Labeling  →  Normal (DG0)  /  Abnormal (DG1)
        │
        ▼
Patient-ID Split  →  Train (70%)  /  Val (15%)  /  Test (15%)
        │
        ▼
EfficientNet-B0 (Transfer Learning)  +  Class Weights  +  Data Augmentation
        │
        ▼
Best Model saved at Epoch 4  →  Val Acc: 70.58%
        │
        ▼
Test Evaluation  →  Accuracy: 71.17%  |  ROC-AUC: 78.12%
        │
        ▼
Single Image Prediction  →  Abnormal (97.01% confidence)
```

### Key Results

| Metric | Value |
|:------:|:-----:|
| Best Validation Accuracy | 70.58% |
| Test Accuracy  | 71.17% |
| Test Precision | 83.42% |
| Test Recall    | 59.22% |
| Test F1 Score  | 69.27% |
| Test ROC-AUC   | 78.12% |
| Total Dataset  | 6,004 images / 188 patients |

### Files

| File | Purpose |
|:----:|:-------:|
| `training.py`   | Model training loop with early stopping |
| `evaluate.py`   | Full test-set evaluation with metrics |
| `predict.py`    | Single image inference |
| `Splitting_data_with_patient_ID.py` | Patient-level train/val/test split |
| `copy_folder.py` | Dataset copying utility |
| `count.py`      | Class distribution counter |
| `renaming_img.py` | Image renaming utility |
| `best_rop_model.pth` | Saved best model weights |

---
*ROP Classification Project — EfficientNet-B0 Transfer Learning on Infant Retinal Database*